# RS-Flow-VQA: Continuous Soft-Prefix Flow Matching & FreeFlow Distillation

This notebook runs the complete aligned-latent pipeline:
1. Clone/update the repository and install it with `uv`.
2. Cache a 4×4 grid of Scale-MAE spatial tokens and Qwen caption IDs.
3. Learn a compact Qwen-compatible caption latent and align image tokens to it.
4. Train the conditional Flow Matching teacher in the compact latent space.
5. Perform target-free conditional FreeFlow distillation.
6. Evaluate multi-reference captions and zero-shot RSVQA-LR transfer.

In [ ]:
# Repository bootstrap. Change REPO_REF if the implementation is on another branch.
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/hugoaslm/rs-flow-vqa.git'
REPO_REF = 'main'  # e.g. 'agent/aligned-latent-pipeline' before the PR is merged
current_dir = Path.cwd()

if (current_dir / 'pyproject.toml').is_file() and (current_dir / 'rs_flow_vqa').is_dir():
    repo_dir = current_dir
    print('Using existing repository:', repo_dir)
else:
    repo_dir = Path('/content/rs-flow-vqa')
    if not (repo_dir / '.git').is_dir():
        subprocess.run(
            ['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, str(repo_dir)],
            check=True,
        )
    else:
        subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(repo_dir), 'checkout', REPO_REF], check=True)
        subprocess.run(
            ['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', REPO_REF],
            check=True,
        )

os.chdir(repo_dir)
assert Path('configs/t4.yaml').is_file(), f'Missing configs/t4.yaml under {Path.cwd()}'
assert Path('pyproject.toml').is_file(), f'Missing pyproject.toml under {Path.cwd()}'
print('Repository ready at:', Path.cwd())
print('Git ref:', subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], text=True).strip())

In [ ]:
# Install uv when needed, then install into the active Colab/Jupyter kernel.
import shutil
import subprocess

if shutil.which('uv') is None:
    subprocess.run('curl -LsSf https://astral.sh/uv/install.sh | sh', shell=True, check=True)
uv_bin = shutil.which('uv') or '/root/.local/bin/uv'
subprocess.run([uv_bin, 'pip', 'install', '--system', '-e', '.[gpu,notebook]'], check=True)

In [ ]:
import torch
from rs_flow_vqa.config import load_config
from rs_flow_vqa.training.train_alignment import train_prompt_autoencoder_pipeline, train_visual_alignment_pipeline
from rs_flow_vqa.training.train_teacher import train_teacher_pipeline
from rs_flow_vqa.training.distill_freeflow import distill_freeflow_pipeline
from rs_flow_vqa.evaluation.eval_caption import evaluate_caption_pipeline
from rs_flow_vqa.evaluation.eval_rsvqa import evaluate_rsvqa_pipeline

print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())

## 1. Choose execution profile
Set `RUN_SMOKE=False` for the real T4 pipeline after placing RSICD and RSVQA-LR under the paths configured in `configs/t4.yaml`.

In [ ]:
RUN_SMOKE = True
cfg = load_config(
    'configs/smoke.yaml' if RUN_SMOKE else 'configs/t4.yaml',
    smoke=RUN_SMOKE,
    device_override='cpu' if RUN_SMOKE else 'cuda',
)
print('Loaded Experiment:', cfg.experiment_name)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
if not RUN_SMOKE:
    assert torch.cuda.is_available(), 'Select a GPU runtime before running the T4 profile.'
    assert torch.cuda.get_device_properties(0).total_memory >= 14 * 1024**3, 'At least 14 GB VRAM is required.'

## 2. Prepare official datasets

When `RUN_SMOKE=False`, the next cell mounts Google Drive, downloads missing official RSICD and RSVQA-LR files, extracts them, and links them into the repository. Downloads, feature caches, and checkpoints persist in Drive. Re-running the cell is safe.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

if RUN_SMOKE:
    print('Smoke profile selected: official dataset download skipped.')
else:
    from google.colab import drive
    drive.mount('/content/drive')

    data_root = Path('/content/drive/MyDrive/rs-flow-vqa-data')
    rsicd_root = data_root / 'RSICD'
    rsvqa_root = data_root / 'RSVQA_LR'
    rsicd_root.mkdir(parents=True, exist_ok=True)
    rsvqa_root.mkdir(parents=True, exist_ok=True)

    def download(url, destination, min_bytes=1):
        destination = Path(destination)
        if destination.exists() and destination.stat().st_size >= min_bytes:
            print('Already downloaded:', destination.name)
            return
        subprocess.run(
            ['wget', '-c', '--show-progress', '-O', str(destination), url],
            check=True,
        )

    # Official RSICD: 10,921 images and dataset_rsicd.json.
    download(
        'https://raw.githubusercontent.com/201528014227051/RSICD_optimal/master/dataset_rsicd.json',
        rsicd_root / 'dataset_rsicd.json',
        min_bytes=1_000_000,
    )
    rsicd_images = rsicd_root / 'RSICD_images'
    if not rsicd_images.exists() or sum(1 for p in rsicd_images.iterdir() if p.is_file()) < 10_000:
        rsicd_archive = data_root / 'RSICD_images.zip'
        download(
            'https://github.com/201528014227051/RSICD_optimal/raw/refs/heads/master/RSICD_images.zip',
            rsicd_archive,
            min_bytes=400_000_000,
        )
        subprocess.run(['unzip', '-q', '-o', str(rsicd_archive), '-d', str(rsicd_root)], check=True)

    # Official RSVQA-LR test split and its 772 Sentinel-2 RGB images.
    zenodo_base = 'https://zenodo.org/records/6344334/files'
    for filename in (
        'LR_split_test_questions.json',
        'LR_split_test_answers.json',
        'LR_split_test_images.json',
    ):
        minimum_size = 100_000 if 'images' in filename else 1_000_000
        download(f'{zenodo_base}/{filename}?download=1', rsvqa_root / filename, minimum_size)
    rsvqa_images = rsvqa_root / 'Images_LR'
    if not rsvqa_images.exists() or sum(1 for p in rsvqa_images.iterdir() if p.is_file()) < 700:
        rsvqa_archive = data_root / 'Images_LR.zip'
        download(f'{zenodo_base}/Images_LR.zip?download=1', rsvqa_archive, 90_000_000)
        subprocess.run(['unzip', '-q', '-o', str(rsvqa_archive), '-d', str(rsvqa_root)], check=True)

    def replace_with_link(destination, source):
        destination = Path(destination)
        source = Path(source)
        source.mkdir(parents=True, exist_ok=True)
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.is_symlink() or destination.is_file():
            destination.unlink()
        elif destination.exists():
            shutil.rmtree(destination)
        destination.symlink_to(source, target_is_directory=True)

    replace_with_link('data/RSICD', rsicd_root)
    replace_with_link('data/RSVQA_LR', rsvqa_root)
    replace_with_link('data/cache_aligned_v3_qwen15b', data_root / 'cache_aligned_v3_qwen15b')
    replace_with_link('outputs/t4_aligned_v3_qwen15b', data_root / 'outputs_t4_aligned_v3_qwen15b')

    # Validate provenance and expected scale before any model download/training.
    from rs_flow_vqa.data.rsicd import RSICDDataset
    from rs_flow_vqa.data.rsvqa import RSVQADataset
    rsicd_check = RSICDDataset('data/RSICD', split='all', is_smoke=False)
    rsvqa_check = RSVQADataset('data/RSVQA_LR', split='test', is_smoke=False)
    print(f'Validated {len(rsicd_check):,} RSICD captions and {len(rsvqa_check):,} RSVQA test questions.')

In [ ]:
# Step 1: Feature caching
from rs_flow_vqa.cli import cache_features_cmd
import argparse

args = argparse.Namespace(
    config='configs/smoke.yaml' if RUN_SMOKE else 'configs/t4.yaml',
    smoke=RUN_SMOKE,
    device='cpu' if RUN_SMOKE else 'cuda',
    seed=42,
    output_dir=cfg.output_dir,
)
cache_features_cmd(args)

In [ ]:
# Step 2: Learn a Qwen-compatible compact prompt latent
# This cell contains the expensive frozen-Qwen backward pass and resumes from Drive.
prompt_ckpt = train_prompt_autoencoder_pipeline(cfg)

In [ ]:
# Step 3: Align Scale-MAE spatial tokens to the prompt latent
visual_ckpt = train_visual_alignment_pipeline(cfg)

In [ ]:
# Step 4: Train compact latent CFM Teacher
teacher_ckpt = train_teacher_pipeline(cfg)

In [ ]:
# Step 5: Target-Free Conditional FreeFlow Distillation
student_ckpt = distill_freeflow_pipeline(cfg)

In [ ]:
# Step 6: Evaluate Caption Quality & Fidelity
cap_metrics = evaluate_caption_pipeline(cfg)

In [ ]:
# Step 7: Evaluate Zero-Shot VQA Transfer on RSVQA-LR
# The T4 profile uses a reproducible 10% subset by default. Set this flag for the full test set.
RUN_FULL_EVAL = False
if RUN_FULL_EVAL:
    cfg.evaluation.rsvqa_subset_fraction = 1.0
rsvqa_metrics = evaluate_rsvqa_pipeline(cfg)

## 3. Summary Metrics Table
Print summary metrics table comparing Baselines, 16-NFE Teacher, and 1-Step FreeFlow Student.

In [ ]:
import pandas as pd

data = [
    {'Model': 'Text-Only Qwen Baseline', 'RSVQA Accuracy (%)': f"{rsvqa_metrics['text_only_baseline']['overall']*100:.2f}%", 'Bridge Latency (ms)': '0.0 ms', 'NFEs': 0},
    {'Model': 'Deterministic Visual Aligner', 'RSVQA Accuracy (%)': f"{rsvqa_metrics['direct_visual_baseline']['overall']*100:.2f}%", 'Bridge Latency (ms)': '-', 'NFEs': 0},
    {'Model': 'CFM Teacher (16-NFE)', 'RSVQA Accuracy (%)': f"{rsvqa_metrics['teacher_16nfe']['overall']*100:.2f}%", 'Bridge Latency (ms)': f"{cap_metrics['teacher_16nfe_latency_ms']:.2f} ms", 'NFEs': 16},
    {'Model': 'FreeFlow Student (1-Step)', 'RSVQA Accuracy (%)': f"{rsvqa_metrics['student_1step']['overall']*100:.2f}%", 'Bridge Latency (ms)': f"{cap_metrics['student_1step_latency_ms']:.2f} ms", 'NFEs': 1},
    {'Model': 'Wrong-Image Control', 'RSVQA Accuracy (%)': f"{rsvqa_metrics['shuffled_image_teacher_control']['overall']*100:.2f}%", 'Bridge Latency (ms)': '-', 'NFEs': 16},
]
df = pd.DataFrame(data)
display(df)
print('Student/teacher latent cosine:', f"{cap_metrics['fidelity_student_vs_teacher32_cosine']:.4f}")
print('Student retains teacher ROUGE-L:', f"{cap_metrics['student_prefix_rouge_l']/max(cap_metrics['teacher_prefix_rouge_l'], 1e-8):.2%}")